# DSA 2026 S2 Assignment

**Member ID:** [INSERT]

**Date:** [INSERT]

---

### How to use this template

The cells below are a skeleton, not an answer sheet. **Add as many cells as you need** — the headings are there so that the marker for each question can find your work, and so that the code cells give you somewhere to start. Nothing stops you splitting one into five, or working somewhere else entirely and pasting the result in.

Two things the brief asks for that this skeleton does not lay out for you:

- a **text cell above** each piece of code, explaining the step you are about to take; and
- a **text cell below** the output, saying what you make of it.

So expect to add text cells throughout — that commentary is a large part of what is being marked, and a notebook of bare code will not score well however good the code is. Keep each question's work under its own heading: different markers take different questions, and an answer that lives somewhere else will not be found.


## Setup

In [3]:
import os, sys, sqlite3, zipfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules


def open_database(name="ctgov.db", archive="ctgov.db.zip"):
    """Locate the database and return its path, extracting the zip if that is all there is.

    Behaves the same locally and on Colab. This runs before sqlite3.connect() on purpose:
    connect() silently creates an empty database when the file is missing, so a missing
    file would otherwise show up much later as 'no such table'.
    """
    places = [os.getcwd()]
    if IN_COLAB:
        places += ["/content", "/content/drive/MyDrive", "/content/drive/MyDrive/DSA"]

    for d in places:
        db = os.path.join(d, name)
        if os.path.isfile(db):
            return db
        archive_path = os.path.join(d, archive)
        if os.path.isfile(archive_path):
            # Unpack to the session disk rather than back into Drive: writing 146 MB to
            # Drive is slow and consumes your quota, and the session copy reads faster.
            target = "/content" if IN_COLAB and d.startswith("/content/drive") else d
            print("Extracting %s to %s ..." % (archive, target))
            try:
                with zipfile.ZipFile(archive_path) as z:
                    z.extractall(target)
            except zipfile.BadZipFile:
                raise RuntimeError(
                    "%s could not be opened. The most likely cause is an interrupted\n"
                    "download - the file should be about 40 MB. Delete it, download it\n"
                    "again, and re-run this cell." % archive_path)
            db = os.path.join(target, name)
            if os.path.isfile(db):
                return db

    looked = "\n  ".join(places)
    if IN_COLAB:
        raise FileNotFoundError(
            "%s not found. Looked in:\n  %s\n\n"
            "On Colab you need to bring the database into the session. Either upload it:\n"
            "    from google.colab import files; files.upload()\n"
            "or mount your Drive and keep it in MyDrive:\n"
            "    from google.colab import drive; drive.mount('/content/drive')\n\n"
            "Uploading is fine for a single sitting; Drive survives a disconnect." % (name, looked)
        )
    raise FileNotFoundError(
        "%s not found. Looked in:\n  %s\n\n"
        "Put %s (or %s) beside this notebook, or change directory to the folder holding it."
        % (name, looked, name, archive)
    )


if tuple(int(x) for x in matplotlib.__version__.split(".")[:2]) < (3, 9):
    print("Warning: matplotlib %s. This notebook needs 3.9 or newer (the box plot below uses\n"
          "tick_labels=). Run:  pip install -U 'matplotlib>=3.9'" % matplotlib.__version__)

DB_PATH = open_database()
conn = sqlite3.connect(DB_PATH)
print("Connected to %s (%.0f MB) - %s environment."
      % (os.path.basename(DB_PATH), os.path.getsize(DB_PATH) / 1e6,
         "Colab" if IN_COLAB else "local"))

# A part-downloaded database opens without complaint and then fails much later with
# something unhelpful like "no such table". Check it here instead, while the cause
# is still obvious.
_integrity = conn.execute("PRAGMA quick_check(1)").fetchone()[0]
_n_tables = conn.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table'").fetchone()[0]
if _integrity != "ok" or _n_tables != 12:
    print("\nThis database does not look complete: %s, %d tables, expected 12.\n"
          "Delete ctgov.db and ctgov.db.zip, download again, and re-run this cell."
          % (_integrity, _n_tables))
else:
    print("%d tables, integrity check ok." % _n_tables)
import seaborn as sns

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", conn)
print("Tables in database: %s" % tables['name'].tolist())

def log_ai(prompt, response, qtag="general", topic="api-call"):
    """Record one AI API call made from your own code, in the format
    compile_ai_logs.py expects (see the AI-use guidance).

    Use this only when your code itself calls an AI model (e.g. an API
    call inside a pipeline). For chat/agent conversations, save the
    exported conversation file into ai_logs/ directly instead.

    qtag: one of "Q1".."Q4", "multiQ", or "general".
    """
    from pathlib import Path
    from datetime import datetime

    log_dir = Path("ai_logs")
    log_dir.mkdir(exist_ok=True)
    stamp = datetime.now().strftime("%Y-%m-%d_%H%M")
    path = log_dir / f"{stamp}_{qtag}_{topic}.md"
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"## Prompt\n\n{prompt}\n\n## Response\n\n{response}\n\n---\n\n")
    return path


def word_count(text):
    """Count the words in a written answer.

    Q1d, Q2d and Q3d are capped at 500 words and Q4c at 300. The brief says
    markers do not read past the limit, so it is worth checking. Paste the
    answer between triple quotes:

        word_count('''My answer goes here.''')
    """
    return len(str(text).split())


def write_manual_note(qtag, topic, note, when=None, log_dir="ai_logs"):
    """Record a conversation you could not save, named like any other log.

    A work tool blocked the export, a temporary chat expired, history was
    cleared, or you simply forgot. None of that is a breach provided you
    record it honestly: say what the conversation covered and why it is
    missing. Falsifying a record, or denying use that occurred, is not the
    same thing at all.

    The same helper is demonstrated in the Getting Started notebook.

        write_manual_note("Q1", "chatgpt-export-blocked",
                          "Talked through the stratified sample for about 15 "
                          "minutes. My work account blocks copying transcripts "
                          "out, so the exchange could not be saved.")
    """
    import os
    from datetime import date

    os.makedirs(log_dir, exist_ok=True)
    stamp = when or date.today().isoformat()          # or pass when="2026-08-14"
    path = os.path.join(log_dir, "%s_%s_%s.md" % (stamp, qtag, topic))
    with open(path, "w", encoding="utf-8") as f:
        f.write("# Manual entry - %s\n\n*%s. Written by me because the conversation "
                "itself could not be saved.*\n\n%s\n"
                % (topic.replace("-", " "), stamp, note))
    print("Wrote", path)
    return path


Connected to ctgov.db (146 MB) - local environment.
12 tables, integrity check ok.
Tables in database: ['consensus_outcomes', 'event_study_results', 'fda_drug_match', 'event_fda_features', 'filing_summary', 'filing_text', 'study_event_link', 'stock_prices', 'studies', 'event_study_enriched', 'asclepius_readouts', 'asclepius_scenario_filings']


# Question 1: Validate and improve the dataset (21 marks)


## Q1a: Evaluate LLM classifications (7 marks)

In [ ]:
# Q1a: Stratified sample (use your member ID as the random seed)
member_id=40100

In [ ]:
# Q1a: LLM outcome vs CAR sign comparison

In [ ]:
# Q1a: Investigate disagreement cases

## Q1b: Construct flagging approach (5 marks)

In [ ]:
# Q1b: Flagging pipeline

In [ ]:
# Q1b: Apply to full dataset and report

## Q1c: Clean and vectorise the data (5 marks)

In [ ]:
# Q1c: Data cleaning

In [ ]:
# Q1c: Word or sentence embeddings (see Supplementary Reading S6)

In [ ]:
# Q1c: Dimensionality reduction

## Q1d: Summary for the IR team (500 words or less, 4 marks)

_Your answer (500 words or less — `word_count()` in the Setup cell will check):_


### AI use — Question 1

_Note here whether and how you used AI for Question 1, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---

# Question 2: Identify comparable trials (20 marks)


## Q2a: Clustering (8 marks)

In [ ]:
# Q2a: Prepare features for clustering

In [ ]:
# Q2a: Clustering algorithm

In [ ]:
# Q2a: Internal validation and comparison of alternatives

## Q2b: Discriminator modelling (4 marks)

In [ ]:
# Q2b: TF-IDF vectorisation of filing text

In [ ]:
# Q2b: Random forest discriminator (default hyperparameters, per-cluster variable importance)

## Q2c: Manual validation (4 marks)

In [ ]:
# Q2c: Manual validation

## Q2d: Summary for the IR team (500 words or less, 4 marks)

_Your answer (500 words or less — `word_count()` in the Setup cell will check):_


### AI use — Question 2

_Note here whether and how you used AI for Question 2, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---

# Question 3: Predict the magnitude of market reactions (27 marks)


## Q3a: Construct feature set (7 marks)

In [ ]:
# Q3a: Load data

In [ ]:
# Q3a: Feature group 1 — Trial and outcome features

In [ ]:
# Q3a: Feature group 2 — Text-based features (at least two representations)

In [ ]:
# Q3a: Feature group 3 — Company/market context features

In [ ]:
# Q3a: Describe feature groups

## Q3b: Construct magnitude prediction models (10 marks)

In [ ]:
# Q3b: Model comparison (at least two model families)

In [ ]:
# Q3b: Ablation analysis (contribution of each feature group)

In [ ]:
# Q3b: Practical discriminatory power (e.g. quintile analysis)

In [ ]:
# Q3b: Data leakage discussion

## Q3c: Asymmetric predictability (6 marks)

In [ ]:
# Q3c: Within-outcome-group analysis

## Q3d: Summary for the IR team (500 words or less, 4 marks)

_Your answer (500 words or less — `word_count()` in the Setup cell will check):_


### AI use — Question 3

_Note here whether and how you used AI for Question 3, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---

# Question 4: Apply your analysis and prepare for deployment (17 marks)


## Q4a: Estimate market reactions for Asclepius's readouts (10 marks)

_Parts (i) to (v) are set out in the brief — work from there, not from these labels. Use the code cells for the work and the markdown cells for what you conclude from it._


**Q4a (i)**

_Your answer:_


In [ ]:
# Q4a (ii): estimates for each readout and scenario


_Your answer:_


In [ ]:
# Q4a (iii): the effect of the features you do not have


_Your answer:_


In [ ]:
# Q4a (iv): workings for the range and its decomposition


**Q4a (iv)**

_Your answer:_


**Q4a (v)**

_Your answer:_


## Q4b: Validate, monitor, and refresh plan (4 marks)

_Written answer — no code required._


_Your answer:_

## Q4c: Advise the IR team on presenting the analysis (300 words or less, 3 marks)

_Your answer (300 words or less — `word_count()` in the Setup cell will check):_


### AI use — Question 4

_Note here whether and how you used AI for Question 4, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---

## Q5: Video presentation (15 marks)

This question is answered as a **video**, not in this notebook — see the brief for what your handover must address, and 'Recording your AI use.docx' for what to submit alongside it.

Before submitting:

- Record your 3–5 minute video and include it with your submission.
- Make sure every dated conversation file is saved under `ai_logs/`, then run the provided `compile_ai_logs.py` script from your submission folder to generate `ai_logs/AI_LOGS.md`. That single file is what you submit.
- If a conversation could not be saved at all, record it honestly with `write_manual_note()` from the Setup cell rather than leaving a silent gap.
- Complete the AI log cell below.
- Paste your YouTube URL into the **Video link** cell at the bottom of this notebook. Without it the marker cannot find your video.

### AI use — Question 5

_Note here whether and how you used AI for Question 5, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---
## Cleanup

In [ ]:
conn.close()
print('Done.')

---

## Video link

The brief requires your YouTube video URL to appear as a hyperlink at the bottom of this notebook. Replace the placeholder below — an unlisted link is fine, a private one is not, because the marker has to be able to open it.

**My video:** [INSERT YOUTUBE URL HERE]
